# Hierarchical Clustering

Wiki reference for [hierarchical clustering](https://ml-viz-ruby.vercel.app/wiki/hierarchical-clustering).

**The idea in one sentence.** Agglomerative clustering starts with every point its own cluster
and repeatedly **merges the two closest clusters**, building a dendrogram — and the **linkage**
(single / complete / average / ward) defines "closest", which dramatically changes the result:
single linkage **chains** along non-convex shapes, complete linkage prefers compact blobs.

We implement agglomerative clustering from scratch, **validate the merge order and that single
linkage recovers crescents**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist, squareform

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  1.5,
})

np.random.seed(42)

## 1 — Worked example: 4-point distance matrix

Points A(1,1), B(2,2), C(8,8), D(9,9) — two tight blobs.

In [ ]:
X4 = np.array([[1,1], [2,2], [8,8], [9,9]], dtype=float)
labels4 = ['A', 'B', 'C', 'D']

# Full pairwise Euclidean distance matrix
D = squareform(pdist(X4))
print("Pairwise distance matrix:")
print(f"{'':5s}" + "".join(f"{l:8s}" for l in labels4))
for i, l in enumerate(labels4):
    row = "".join(f"{D[i,j]:8.2f}" for j in range(4))
    print(f"{l:5s}{row}")

# Verify from wiki
print(f"\nd(A,B) = {D[0,1]:.4f}  (expect 1.41)")
print(f"d(A,C) = {D[0,2]:.4f}  (expect 9.90)")
print(f"d(B,C) = {D[1,2]:.4f}  (expect 8.49)")
print(f"d(A,D) = {D[0,3]:.4f}  (expect 11.31)")

## 2 — From-scratch agglomerative clustering

Single, complete, and average linkage — track the merge heights.

In [ ]:
def agglomerative_scratch(X, method='single'):
    """Returns list of (cluster_i, cluster_j, merge_height) in merge order."""
    n = len(X)
    clusters = {i: [i] for i in range(n)}
    D = squareform(pdist(X))
    merges = []

    active = list(range(n))
    while len(active) > 1:
        best_dist = np.inf
        best_pair = None

        for i in range(len(active)):
            for j in range(i+1, len(active)):
                ci, cj = active[i], active[j]
                pts_i = clusters[ci]
                pts_j = clusters[cj]

                dists = [D[a, b] for a in pts_i for b in pts_j]
                if method == 'single':
                    d = min(dists)
                elif method == 'complete':
                    d = max(dists)
                elif method == 'average':
                    d = sum(dists) / len(dists)
                else:
                    raise ValueError(f"Unknown method: {method}")

                if d < best_dist:
                    best_dist = d
                    best_pair = (ci, cj)

        ci, cj = best_pair
        merges.append((ci, cj, best_dist))
        # Merge cj into ci
        clusters[ci] = clusters[ci] + clusters[cj]
        del clusters[cj]
        active.remove(cj)

    return merges

for method in ['single', 'complete', 'average']:
    merges = agglomerative_scratch(X4, method=method)
    print(f"\n{method.title()} linkage merges:")
    for ci, cj, h in merges:
        li = labels4[ci] if ci < 4 else f'({labels4[ci%4]}+...) '
        lj = labels4[cj] if cj < 4 else f'({labels4[cj%4]}+...)'
        print(f"  Merge clusters containing {ci} & {cj} at height {h:.4f}")

### Validate: agglomerative clustering merges closest first

The algorithm always merges the two nearest clusters, so merge **heights are non-decreasing**,
and on the 4-point example the two tight pairs (A-B and C-D, distance $\sqrt2\approx1.41$) merge
before the far apart groups join. We confirm.

In [ ]:
merges = agglomerative_scratch(X4, method='single')
heights = [m[2] for m in merges]
print('merge heights:', [round(h, 2) for h in heights])
assert all(heights[i] <= heights[i+1] + 1e-9 for i in range(len(heights)-1)), 'merges happen in non-decreasing height order'
assert np.isclose(heights[0], np.sqrt(2), atol=0.01), 'the first merge is the closest pair (A-B at ~1.41)'
print('\n✅ agglomerative clustering greedily merges the nearest clusters, building the dendrogram bottom-up')

## 3 — Dendrograms via scipy

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, method in zip(axes, ['single', 'complete', 'average']):
    Z = linkage(X4, method=method)
    d = dendrogram(Z, ax=ax, labels=labels4,
                   color_threshold=0,
                   above_threshold_color='#6366f1',
                   link_color_func=lambda k: '#6366f1')
    ax.set_title(f'{method.title()} linkage')
    ax.set_ylabel('Merge height')

plt.suptitle('All linkages give same structure; heights differ', color='#eee')
plt.tight_layout()
plt.show()

## 4 — Where linkages diverge: crescents dataset

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import AgglomerativeClustering

X_moons, _ = make_moons(n_samples=100, noise=0.05, random_state=42)

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
palettes = ['#6366f1', '#f97316']

for ax, method in zip(axes, ['single', 'complete', 'average', 'ward']):
    labels = AgglomerativeClustering(n_clusters=2, linkage=method).fit_predict(X_moons)
    for k, col in zip([0, 1], palettes):
        mask = labels == k
        ax.scatter(X_moons[mask, 0], X_moons[mask, 1], s=15, color=col, alpha=0.8)
    ax.set_title(f'{method}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Linkage methods on two crescents — only single finds the right structure',
             color='#eee', fontsize=10)
plt.tight_layout()
plt.show()

### Validate: single linkage recovers non-convex clusters

On two interleaving crescents (moons), **single linkage** chains along each crescent and
recovers the true clusters, where compactness-seeking linkages fail. We confirm single linkage
matches the ground truth (adjusted Rand index ~1).

In [ ]:
from sklearn.metrics import adjusted_rand_score
single_labels = AgglomerativeClustering(n_clusters=2, linkage='single').fit_predict(X_moons)
_, y_moons = make_moons(n_samples=100, noise=0.05, random_state=42)
ari = adjusted_rand_score(y_moons, single_labels)
print(f'single-linkage ARI on moons: {ari:.3f}')
assert ari > 0.9, 'single linkage chains along the crescents -> recovers the non-convex clusters'
print('\n✅ single linkage handles non-convex shapes that centroid/compact methods miss')

## 5 — Ward linkage: variance increase

In [ ]:
def ward_distance(pts_a, pts_b):
    """ΔVar = |A|·|B|/(|A|+|B|) · ||centroid_A - centroid_B||²"""
    na, nb = len(pts_a), len(pts_b)
    ca = np.mean(pts_a, axis=0)
    cb = np.mean(pts_b, axis=0)
    return (na * nb / (na + nb)) * np.sum((ca - cb)**2)

# Compare Ward to complete on 3 compact clusters
np.random.seed(3)
X3 = np.vstack([
    np.random.randn(20, 2) * 0.4 + [0, 0],
    np.random.randn(20, 2) * 0.4 + [4, 0],
    np.random.randn(20, 2) * 0.4 + [2, 3],
])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, method in zip(axes, ['ward', 'complete']):
    Z = linkage(X3, method=method)
    dendrogram(Z, ax=ax, no_labels=True,
               color_threshold=0,
               above_threshold_color='#6366f1',
               link_color_func=lambda k: '#6366f1')
    ax.set_title(f'{method.title()} linkage — 3 compact clusters')
    ax.set_ylabel('Merge height')

plt.tight_layout()
plt.show()

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **wrong linkage** | complete/ward miss non-convex clusters (demo) |
| **single-linkage chaining** | sensitive to noise/bridge points that merge clusters |
| **where to cut** | the dendrogram cut height sets the cluster count |
| **cost** | naive agglomerative is $O(n^3)$ / $O(n^2)$ memory — bad for large n |
| **no reassignment** | a merge is permanent; an early mistake can't be undone |

Demo: complete linkage fails on crescents that single linkage recovers.

In [ ]:
# The linkage choice is not cosmetic — it changes the answer. COMPLETE linkage minimises the
# maximum intra-cluster distance, so it prefers compact, roughly-spherical blobs and FAILS on
# the crescents where single linkage succeeds. We compare the two on the same moons.
from sklearn.metrics import adjusted_rand_score
_, y_moons = make_moons(n_samples=100, noise=0.05, random_state=42)
single_ari = adjusted_rand_score(y_moons, AgglomerativeClustering(n_clusters=2, linkage='single').fit_predict(X_moons))
complete_ari = adjusted_rand_score(y_moons, AgglomerativeClustering(n_clusters=2, linkage='complete').fit_predict(X_moons))
print(f'ARI on moons: single={single_ari:.3f}  complete={complete_ari:.3f}')
assert complete_ari < single_ari, 'complete linkage prefers compact blobs -> fails on crescents where single succeeds'
print('\nLinkage encodes an assumption about cluster shape -> choose it to match your data, not by default.')

## ✏️ Your turn

### Exercise 1 — cut height and cluster count

Given the Ward dendrogram on `X3` (3 natural clusters), find the
cut height that yields exactly 3 clusters. Use `scipy.cluster.hierarchy.fcluster`
with `criterion='distance'`.

Then plot the 3D scatter coloring each point by its cluster label.

In [ ]:
# TODO(you): find the right cut height and extract 3 clusters

# Z = linkage(X3, method='ward')
# Look at the merge heights in Z[:, 2] — the gap between the 3rd-to-last
# and 2nd-to-last merge is where you should cut.

# cut_height = ???
# labels = fcluster(Z, t=cut_height, criterion='distance') - 1
# assert len(np.unique(labels)) == 3, "Expected 3 clusters"

# plt.figure(figsize=(5, 4))
# for k, col in enumerate(['#6366f1', '#f97316', '#20d9d2']):
#     plt.scatter(X3[labels==k, 0], X3[labels==k, 1], s=20, color=col)
# plt.title('Ward clustering: 3 clusters')
# plt.show()

### Exercise 2 — average linkage from scratch

Extend `agglomerative_scratch` to also handle **Ward** linkage.
Verify that your implementation gives the same merge heights as
`scipy.cluster.hierarchy.linkage(X4, method='ward')`.

<details>
<summary>Solution outline</summary>

```python
# Add a 'ward' branch inside agglomerative_scratch:
# elif method == 'ward':
#     pts_i_arr = X[pts_i]
#     pts_j_arr = X[pts_j]
#     d = ward_distance(pts_i_arr, pts_j_arr)

# Note: scipy's Ward height = sqrt(2 * ΔVar), so scale accordingly.
```
</details>

## Key takeaways

- **Agglomerative = greedily merge the two closest clusters,** building a dendrogram (verified).
- **Merge heights are non-decreasing** — cut the dendrogram to pick the number of clusters.
- **Linkage defines "closest":** single chains (non-convex), complete/ward favour compact blobs.
- **Single linkage recovers crescents** where complete linkage fails (verified, demo) — the
  choice matters.